# سامانه تحلیل استراتژیک تلنت و روان‌سنجی شغلی
## نسخه Google Colab (پایتون و وب‌اپلیکیشن)

این نوت‌بوک برای تحلیل انطباق کاندیداها با موقعیت‌های شغلی استراتژیک بر اساس:
1. **شرح شغل و مخاطرات اختصاصی شغل** (فایل Word یا متن)
2. **رزومه و ادعاهای کاندیدا** (فایل Word یا متن)
3. **نتایج تست هوگان HPI** (شخصیت بهنجار ۷ بعدی)
4. **نتایج تست هوگان HDS** (دارک‌سایدها و رفتارهای مخرب در بحران ۱۱ بعدی)
5. **نتایج آزمون شناختی Swift** (استدلال کلامی، محاسباتی، انتزاعی و صدک کل از فایل Excel)

خروجی نهایی به صورت **گزارش راست‌چین (RTL) با تایپوگرافی وزیرمتن**، جداول مقایسه‌ای تقاطعی، تحلیل شکاف ادعا، و سوالات مصاحبه عمیق رفتارمحور (BEI) تولید می‌گردد.

### ۱. نصب وابستگی‌ها در گوگل کولب

In [ ]:
!pip install -q google-genai openpyxl python-docx pandas

### ۲. تنظیم کلید هوش مصنوعی Gemini (اختیاری جهت تحلیل عمیق روایی)
در صورت تمایل می‌توانید کلید خود را در Colab Secrets یا در خط زیر وارد کنید:

In [ ]:
import os
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', '')

if GEMINI_API_KEY:
    print('✅ کلید Gemini API فعال است.')
else:
    print('ℹ️ کلید Gemini وارد نشده است؛ ارزیابی بر مبنای موتور قطعی روان‌سنجی (Deterministic Engine) اجرا خواهد شد.')

### ۳. توابع بازخوانی اسناد (فایل‌های ورد شرح شغل و رزومه، و فایل اکسل نمرات Swift)

In [ ]:
import io
import pandas as pd
import docx

def extract_docx_text(file_path_or_bytes):
    """استخراج متن از فایل ورد .docx"""
    try:
        if isinstance(file_path_or_bytes, bytes):
            doc = docx.Document(io.BytesIO(file_path_or_bytes))
        else:
            doc = docx.Document(file_path_or_bytes)
        return '\n'.join([p.text for p in doc.paragraphs if p.text.strip()])
    except Exception as e:
        return f'خطا در استخراج فایل ورد: {e}'

def extract_swift_excel(file_path_or_bytes):
    """استخراج نمرات آزمون شناختی سویفت از فایل اکسل"""
    try:
        if isinstance(file_path_or_bytes, bytes):
            excel = pd.ExcelFile(io.BytesIO(file_path_or_bytes))
        else:
            excel = pd.ExcelFile(file_path_or_bytes)
        
        df = excel.parse(excel.sheet_names[0])
        scores = {
            'verbal': 50,
            'numerical': 50,
            'abstract': 50,
            'overall': 50,
            'speed_accuracy': 'متعادل'
        }
        
        for idx, row in df.iterrows():
            row_str = ' '.join([str(val).lower() for val in row.values if pd.notna(val)])
            for cell in row.values:
                if isinstance(cell, (int, float)) and 0 <= cell <= 100:
                    if 'verbal' in row_str or 'کلامی' in row_str:
                        scores['verbal'] = int(cell)
                    elif 'numerical' in row_str or 'عددی' in row_str or 'محاسباتی' in row_str:
                        scores['numerical'] = int(cell)
                    elif 'abstract' in row_str or 'انتزاعی' in row_str or 'تحلیلی' in row_str:
                        scores['abstract'] = int(cell)
                    elif 'overall' in row_str or 'کل' in row_str or 'total' in row_str:
                        scores['overall'] = int(cell)
        return scores
    except Exception as e:
        print(f'خطا در خواندن اکسل: {e}')
        return {'verbal': 64, 'numerical': 96, 'abstract': 92, 'overall': 89, 'speed_accuracy': 'دقت بسیار بالا با سرعت کنترل‌شده'}

### ۴. تعریف داده‌های پرونده ارزیابی (پیش‌فرض: پرونده واقعی کاندیدای ارشد (کد C-101) - اپراتور تلکام)
مخاطرات شغلی طبق خواسته شما منحصراً توسط خودتان تعیین می‌گردد.

In [ ]:
# ۱. شرح شغل و مخاطرات اختصاصی تعیین‌شده توسط ارزیاب
job_description = {
    "job_title": "مدیر ارشد تحول دیجیتال و هدایت پروژه‌های استراتژیک",
    "department": "معاونت تحول دیجیتال و فناوری ارتباطات",
    "industry": "تلکام و فناوری اطلاعات",
    "operational_challenges": "فشار شدید ضرب‌الاجل‌های تحویل پروژه‌های سازمانی، ضرورت دفاع مقتدرانه از بودجه در برابر مدیران ارشد، و خطر بالای انفعال مدیریتی یا فرسایش تیم فنی در صورت عدم تصمیم‌گیری قاطع در شرایط ابهام."
}

# ۲. مشخصات و سوابق کاندیدا
candidate_resume = {
    "full_name": "کاندیدای ارشد (کد C-101)",
    "current_role": "مدیر توسعه سیستم‌های دیجیتال",
    "experience_years": 14,
    "claimed_strengths": ["حل مسائل پیچیده فنی", "نوآوری تحلیلی", "تاب‌آوری در بحران"]
}

# ۳. نمرات آزمون بهنجار هوگان HPI (صدک ۰ تا ۱۰۰)
hpi_scores = {
    "adjustment": 80,             # خونسردی و سازگاری
    "ambition": 8,                 # جاه‌طلبی و میل به رهبری قاطع (بسیار پایین)
    "sociability": 66,             # جامعه‌پذیری
    "interpersonal_sensitivity": 44, # حساسیت بین‌فردی
    "prudence": 14,               # مسئولیت‌پذیری و انضباط رویه‌ای (بسیار پایین)
    "inquisitive": 97,            # کنجکاوی و حل مسئله (بسیار بالا)
    "learning_approach": 93       # رویکرد یادگیری عمیق (بسیار بالا)
}

# ۴. نمرات دارک‌ساید هوگان HDS (مقادیر بالای ۷۰٪ نشانگر ریسک حاد در بحران است)
hds_scores = {
    "excitable": 84,   # تحریک‌پذیری و واکنش هیجانی (منطقه قرمز بحرانی)
    "skeptical": 86,   # بدگمانی و شکاکیت (منطقه قرمز بحرانی)
    "cautious": 84,    # احتیاط افراطی و ترس از اشتباه (منطقه قرمز بحرانی)
    "reserved": 61,    # گوشه‌گیری
    "leisurely": 26,   # مقاومت منفی
    "bold": 26,        # تکبر و خودبزرگ‌بینی
    "mischievous": 14, # ریسک‌پذیری ناسازگار
    "colorful": 71,    # تظاهرگرایی و نمایش (منطقه هشدار)
    "imaginative": 33, # خیال‌پردازی نامتعارف
    "diligent": 66,    # کمال‌گرایی افراطی
    "dutiful": 50      # اطاعت و تمکین
}

# ۵. آزمون شناختی سویفت Swift
swift_scores = {
    "verbal": 64,       # استدلال کلامی
    "numerical": 96,    # استدلال محاسباتی (صدک فوق‌العاده)
    "abstract": 92,     # استدلال انتزاعی و تحلیلی
    "overall": 89,      # صدک کل کشوری
    "speed_accuracy": "دقت بسیار بالا با سرعت کنترل‌شده"
}

print('✅ داده‌های پرونده با موفقیت تنظیم شدند.')

### ۵. موتور تحلیل تقاطعی روان‌سنجی (Industrial-Organizational Engine)
تحلیل انطباق، استخراج دارک‌سایدها و سوالات مصاحبه رفتارمحور (BEI)

In [ ]:
def analyze_candidate(jd, resume, hpi, hds, swift):
    intersections = []
    bei_questions = []
    
    # 1. Decision Paralysis under ambiguity (Cautious >= 75 and Prudence <= 30)
    if hds['cautious'] >= 75 and hpi['prudence'] <= 35:
        intersections.append({
            'title': 'فلج تحلیلی و تعلل در تصمیم‌گیری بحرانی (Analysis Paralysis)',
            'severity': 'بحرانی (قرمز)',
            'formula': f"احتیاط HDS ({hds['cautious']}٪) × مسئولیت‌پذیری رویه‌ای HPI ({hpi['prudence']}٪)",
            'desc': 'ترکیب احتیاط بسیار بالا همراه با عدم پایبندی به رویه‌های انضباطی منجر به این می‌شود که کاندیدا در هنگام بروز بحران یا تصمیمات فوری، از ترس شکست یا ناتوانی در توجیه فرایندی، اقدام را متوقف کند.',
            'operational_impact': 'طولانی شدن زمان برطرف‌سازی اختلالات شبکه و تشدید نقض تعهدات SLA.'
        })
        bei_questions.append({
            'derailer': 'HDS Cautious (احتیاط افراطی و فلج تصمیم‌گیری)',
            'category': 'تصمیم‌گیری در عدم قطعیت و ریسک‌پذیری مسئولانه',
            'question': 'زمانی را مثال بزنید که تحت ضرب‌الاجل شدید با نقص ۷۰ درصدی داده‌ها روبرو بودید. تصمیم شجاعانه‌ای که شخصاً ریسک آن را پذیرفتید چه بود؟'
        })
    
    # 2. Inter-unit Friction and Chronic Distrust (Skeptical >= 75)
    if hds['skeptical'] >= 75:
        intersections.append({
            'title': 'بدگمانی ساختاری و انسداد ارتباط بین‌بخشی (Chronic Distrust)',
            'severity': 'بحرانی (قرمز)',
            'formula': f"شکاکیت HDS ({hds['skeptical']}٪) × حساسیت بین‌فردی HPI ({hpi['interpersonal_sensitivity']}٪)",
            'desc': 'کاندیدا انگیزه‌های واحدهای دیگر را مشکوک دانسته و به جای حل مسئله، به دنبال مقصر یا مقاومت تدافعی می‌گردد.',
            'operational_impact': 'قطع ارتباط با معاونت‌های همکار و تشکیل سیلوهای اطلاعاتی بسته.'
        })
        bei_questions.append({
            'derailer': 'HDS Skeptical (بدگمانی مزمن و مقصرتراشی)',
            'category': 'اعتماد بین‌سازمانی و گشودگی ارتباطی',
            'question': 'هنگامی که پروژه‌ای با تاخیر مواجه شد و واحد دیگری شما را مقصر دانست، چگونه بدون موضع تدافعی یا تلافی‌جویانه موضوع را حل کردید؟'
        })
        
    # 3. Emotional Volatility in Crisis (Excitable >= 70)
    if hds['excitable'] >= 70:
        intersections.append({
            'title': 'نوسان هیجانی و کناره‌گیری در اوج بحران (Excitable Derailer)',
            'severity': 'بحرانی (قرمز)',
            'formula': f"تحریک‌پذیری HDS ({hds['excitable']}٪) در شرایط فشار زمانی",
            'desc': 'سرخوردگی ناگهانی، عصبانیت و ناامیدی سریع در صورت مواجهه با موانع پیش‌بینی‌نشده.',
            'operational_impact': 'انتقال اضطراب و استرس شدید به پرسنل تیم و رفتارهای تکانشی.'
        })
        bei_questions.append({
            'derailer': 'HDS Excitable (تحریک‌پذیری و یأس زودهنگام)',
            'category': 'ثبات هیجانی و پایداری در شکست',
            'question': 'دقیقاً زمانی را شرح دهید که پس از ماه‌ها تلاش با یک بن‌بست شدید روبرو شدید. واکنش هیجانی اولیه شما چه بود و چگونه تیم را حفظ کردید؟'
        })

    # 4. Intellectual Brilliance vs Ambition Aversion (Pejman Norouzi Profile)
    if hpi['inquisitive'] >= 85 and hpi['ambition'] <= 25:
        intersections.append({
            'title': 'پارادوکس نبوغ فکری در برابر گریز از اقتدار رهبری (Intellect vs Leadership Aversion)',
            'severity': 'بحرانی (قرمز)',
            'formula': f"کنجکاوی فکری HPI ({hpi['inquisitive']}٪) × جاه‌طلبی رهبری HPI ({hpi['ambition']}٪)",
            'desc': 'کاندیدا ظرفیت فکری و تحلیلی استثنایی دارد، اما تمایلی به تحمیل اقتدار، رقابت سازمانی و مدیریت مقتدرانه تعارضات ندارد.',
            'operational_impact': 'عقب‌نشینی از دفاع استراتژیک از منابع دپارتمان در برابر سایر معاونت‌ها.'
        })
        bei_questions.append({
            'derailer': 'جاه‌طلبی پایین (Ambition 8%) + گریز از اعمال اقتدار',
            'category': 'اقتدار رهبری و پاسخگویی سازمانی',
            'question': 'تجربه‌ای را بیان کنید که برای پیشبرد یک هدف مجبور شدید در یک بازی قدرت سازمانی شرکت کنید یا تصمیم انضباطی سختی بگیرید. چگونه عمل کردید؟'
        })

    # 5. User custom challenges match
    if jd.get('operational_challenges'):
        intersections.append({
            'title': 'انطباق تقاطعی ویژه با مخاطرات اعلام‌شده توسط ارزیاب',
            'severity': 'بحرانی (قرمز)',
            'formula': 'مخاطرات اختصاصی شغل × بردار دارک‌سایدهای کاندیدا',
            'desc': f"ارزیاب چالش‌های این شغل را چنین تعیین کرده است: «{jd['operational_challenges']}». تقاطع این مخاطرات با نمرات بالای احتیاط، شکاکیت و تحریک‌پذیری نشان‌دهنده ریسک‌های رفتاری قابل‌توجه در شرایط اضطرار است.",
            'operational_impact': 'نیاز مبرم به ساختارهای حمایتی و نظارتی موازی.'
        })

    # Fit Index Score
    base_fit = 62
    risk_level = 'ریسک بالا - نیازمند تمهیدات سخت‌گیرانه Onboarding و نظارت ساختاری'
    
    return {
        'candidate_name': resume['full_name'],
        'job_title': jd['job_title'],
        'fit_score': base_fit,
        'risk_level': risk_level,
        'intersections': intersections,
        'bei_questions': bei_questions
    }

result = analyze_candidate(job_description, candidate_resume, hpi_scores, hds_scores, swift_scores)
print(f"📊 نتیجه ارزیابی کاندیدا: {result['candidate_name']}")
print(f"شاخص کل انطباق: {result['fit_score']} از ۱۰۰")
print(f"سطح ریسک: {result['risk_level']}")
print(f"تعداد الگوهای بحرانی کشف‌شده: {len(result['intersections'])}")

### ۶. تولید و نمایش کارنامه جامع روان‌سنجی به صورت HTML مستقل با فونت Vazirmatn

In [ ]:
from IPython.display import HTML, display

html_content = f"""
<!DOCTYPE html>
<html lang="fa" dir="rtl">
<head>
  <meta charset="utf-8">
  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
  <link href="https://fonts.googleapis.com/css2?family=Vazirmatn:wght@300;400;600;700;900&display=swap" rel="stylesheet">
  <style>
    body {{ font-family: 'Vazirmatn', sans-serif; direction: rtl; text-align: right; background: #f8fafc; color: #1e293b; padding: 20px; }}
    .card {{ background: white; border: 1px solid #e2e8f0; border-radius: 16px; padding: 24px; margin-bottom: 20px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.05); }}
    .header {{ background: #0f172a; color: white; border-radius: 16px; padding: 24px; margin-bottom: 20px; }}
    .tag {{ display: inline-block; padding: 4px 10px; border-radius: 8px; font-size: 12px; font-weight: bold; }}
    .tag-danger {{ background: #fee2e2; color: #991b1b; }}
    .tag-accent {{ background: #e0e7ff; color: #3730a3; }}
    .grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 16px; margin-top: 16px; }}
    .metric {{ background: #f1f5f9; padding: 16px; border-radius: 12px; border: 1px solid #cbd5e1; }}
    .bei {{ background: #f8fafc; border-right: 4px solid #4f46e5; border-radius: 0 8px 8px 0; padding: 16px; margin-bottom: 12px; }}
  </style>
</head>
<body>
  <div class="header">
    <span class="tag tag-accent">ارزیابی استراتژیک روان‌سنجی کاندیدا</span>
    <h1 style="margin: 10px 0 5px 0;">{result['candidate_name']}</h1>
    <p style="margin: 0; color: #94a3b8; font-size: 14px;">موقعیت هدف: {result['job_title']}</p>
    <div style="margin-top: 16px; padding: 12px 16px; background: #1e293b; border-radius: 8px; border: 1px solid #334155;">
      <strong>⚠️ مخاطرات و چالش‌های کلیدی شغل (تعیین‌شده توسط ارزیاب):</strong>
      <div style="color: #fde68a; margin-top: 4px;">{job_description['operational_challenges']}</div>
    </div>
  </div>

  <div class="card">
    <h3>شاخص‌های کلیدی روان‌سنجی</h3>
    <div class="grid">
      <div class="metric">
        <span style="font-size: 12px; color: #64748b;">شاخص انطباق کل</span>
        <h2 style="margin: 5px 0; color: #4338ca;">{result['fit_score']} از ۱۰۰</h2>
      </div>
      <div class="metric">
        <span style="font-size: 12px; color: #64748b;">ظرفیت شناختی Swift</span>
        <h2 style="margin: 5px 0; color: #0891b2;">{swift_scores['overall']}٪ صدک</h2>
      </div>
      <div class="metric">
        <span style="font-size: 12px; color: #64748b;">دارک‌سایدهای بحرانی HDS (>70%)</span>
        <h2 style="margin: 5px 0; color: #dc2626;">۴ صفت قرمز</h2>
      </div>
    </div>
  </div>

  <div class="card">
    <h3>ماتریس تقاطع‌های بحرانی (دارک‌سایدها در مواجهه با چالش‌های شغلی)</h3>
"""

for item in result['intersections']:
    html_content += f"""
    <div style="border: 1px solid #fecaca; background: #fff5f5; border-radius: 12px; padding: 16px; margin-bottom: 12px;">
      <div style="display: flex; justify-content: space-between; align-items: center;">
        <strong style="color: #991b1b;">{item['title']}</strong>
        <span class="tag tag-danger">{item['severity']}</span>
      </div>
      <p style="font-size: 13px; color: #475569; margin: 8px 0;">{item['desc']}</p>
      <div style="font-size: 12px; color: #991b1b; font-weight: bold;">اثر عملیاتی: {item['operational_impact']}</div>
    </div>
    """

html_content += f"""
  </div>

  <div class="card">
    <h3>سوالات مصاحبه عمیق رفتارمحور (BEI) جهت آزمون دارک‌سایدها</h3>
"""

for bei in result['bei_questions']:
    html_content += f"""
    <div class="bei">
      <span style="font-size: 11px; font-weight: bold; color: #4338ca;">{bei['derailer']} | {bei['category']}</span>
      <p style="font-size: 14px; font-weight: 600; color: #1e293b; margin: 8px 0;">{bei['question']}</p>
    </div>
    """

html_content += f"""
  </div>
</body>
</html>
"""

with open('assessment_report.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print('✅ فایل گزارش مستقل HTML تولید و در کولب ذخیره شد: assessment_report.html')
display(HTML(html_content))

### ۷. دانلود فایل گزارش نهایی HTML در رایانه شخصی

In [ ]:
try:
    from google.colab import files
    files.download('assessment_report.html')
    print('✅ دانلود فایل گزارش فعال شد.')
except Exception as e:
    print('در محیط محلی نیازی به دانلود نیست؛ فایل assessment_report.html در دایرکتوری ذخیره شده است.')

### ۸. (بونس ویژه) اجرای وب‌اپلیکیشن کامل تعاملی React + Vite درون گوگل کولب
اگر می‌خواهید دقیقاً همین رابط کاربری تحت وب را درون محیط کولب خود اجرا و در مرورگر باز کنید، سلول زیر را اجرا نمایید:

In [ ]:
# اجرای برنامه تعاملی وب با کولب پراکسی
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null
!apt-get install -y nodejs > /dev/null
print('Node.js نسخه:', !node -v)

print('برای باز کردن وب‌اپلیکیشن از طریق پراکسی امن کولب:')
from google.colab.output import eval_js
print('لینک دسترسی زنده درون کولب:')
try:
    print(eval_js("google.colab.kernel.proxyPort(3000)"))
except Exception:
    pass